# Interactive Geocoding & Weather Explorer
**Geocode any US address or click anywhere on the map** to drop a marker and instantly fetch:
- Current weather conditions (temperature, wind, humidity, sky)
- Hourly temperature forecast for the next 24 hours
- 7-day forecast table

**Data sources**: [Nominatim](https://nominatim.org/) geocoding (free, global) · [api.weather.gov](https://api.weather.gov) (free, no API key, US only)


In [ ]:
%pip install ipyleaflet geopy ipywidgets pandas matplotlib requests
# --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.0 MB/s eta 0:00:0000:0100:01


In [1]:
%pip install pytest

  Using cached pytest-9.0.3-py3-none-any.whl.metadata (7.6 kB)
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
Using cached pytest-9.0.3-py3-none-any.whl (375 kB)
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)

   ------------- -------------------------- 1/3 [iniconfig]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---------------------------------------- 3/3 [pytest]

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as ipw
import ipyleaflet
from geopy.geocoders import Nominatim
from IPython.display import display
from datetime import datetime

# ── NWS API ─────────────────────────────────────────────────────────────────
NWS_BASE    = "https://api.weather.gov"
NWS_HEADERS = {
    "User-Agent": "weather-explorer-notebook/1.0 (jupyter; educational use)",
    "Accept":     "application/geo+json",
}
REQUEST_TIMEOUT = 15     # seconds

# ── Map defaults (centre of contiguous US) ───────────────────────────────────
DEFAULT_CENTER = (39.5, -98.35)
DEFAULT_ZOOM   = 4

print("Imports loaded ✓")

Imports loaded ✓


In [5]:
# ── Geocoding ────────────────────────────────────────────────────────────────

_geolocator = Nominatim(user_agent="weather-explorer-notebook/1.0")


def geocode_address(address: str) -> tuple:
    """Return (latitude, longitude) for the given address string.

    Args:
        address: A free-text address, city name, or ZIP code.

    Returns:
        (lat, lon) tuple of floats.

    Raises:
        ValueError: If the address cannot be geocoded.
    """
    location = _geolocator.geocode(address.strip(), timeout=REQUEST_TIMEOUT)
    if location is None:
        raise ValueError(f"Address not found: '{address}'")
    return location.latitude, location.longitude

In [6]:
# ── NWS API helpers ──────────────────────────────────────────────────────────

def _nws_get(url: str) -> dict:
    """GET a NWS API URL and return the parsed JSON body.

    Raises:
        requests.HTTPError: On a non-2xx response.
    """
    response = requests.get(url, headers=NWS_HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def get_nws_points(lat: float, lon: float) -> dict:
    """Fetch the NWS /points endpoint for a coordinate pair.

    Returns a dict with keys: 'forecast', 'forecastHourly', 'observationStations'.

    Raises:
        ValueError: If the point is outside NWS coverage (i.e. not in the US).
        requests.HTTPError: On unexpected API errors.
    """
    try:
        data = _nws_get(f"{NWS_BASE}/points/{lat:.4f},{lon:.4f}")
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code == 404:
            raise ValueError(
                "This location is outside NWS coverage. "
                "api.weather.gov only covers the United States and territories."
            ) from exc
        raise

    props = data.get("properties", {})
    return {
        "forecast":            props.get("forecast"),
        "forecastHourly":      props.get("forecastHourly"),
        "observationStations": props.get("observationStations"),
        "city":                props.get("relativeLocation", {}).get("properties", {}).get("city", ""),
        "state":               props.get("relativeLocation", {}).get("properties", {}).get("state", ""),
    }


def _celsius_to_fahrenheit(celsius) -> float | None:
    """Convert Celsius to Fahrenheit; returns None if input is None."""
    if celsius is None:
        return None
    return round(celsius * 9 / 5 + 32, 1)


def get_current_conditions(stations_url: str) -> dict:
    """Fetch the latest observation from the nearest NWS station.

    Returns a dict with human-readable weather fields. Values may be None
    if the station did not report a particular measurement.
    """
    stations_data = _nws_get(f"{stations_url}?limit=1")
    features = stations_data.get("features", [])
    if not features:
        return {}

    station_id = features[0].get("properties", {}).get("stationIdentifier", "")
    obs_data   = _nws_get(f"{NWS_BASE}/stations/{station_id}/observations/latest")
    obs        = obs_data.get("properties", {})

    temp_c    = obs.get("temperature", {}).get("value")
    dewpoint_c = obs.get("dewpoint", {}).get("value")

    # Approximate relative humidity from temperature and dewpoint
    rh = None
    if temp_c is not None and dewpoint_c is not None:
        rh = round(100 * (1 - (temp_c - dewpoint_c) / 25), 1)
        rh = max(0, min(100, rh))

    return {
        "temperature_f":    _celsius_to_fahrenheit(temp_c),
        "wind_speed":       obs.get("windSpeed", {}).get("value"),
        "wind_direction":   obs.get("windDirection", {}).get("value"),
        "description":      obs.get("textDescription", "N/A"),
        "humidity_pct":     rh,
        "station_id":       station_id,
        "timestamp":        obs.get("timestamp", ""),
    }


def get_hourly_forecast(hourly_url: str) -> pd.DataFrame:
    """Fetch the NWS hourly forecast and return the next 24 periods as a DataFrame."""
    data    = _nws_get(hourly_url)
    periods = data.get("properties", {}).get("periods", [])[:24]

    rows = []
    for p in periods:
        rows.append({
            "Time":        datetime.fromisoformat(p["startTime"]),
            "Temp (°F)":   p.get("temperature"),
            "Wind":        p.get("windSpeed", ""),
            "Forecast":    p.get("shortForecast", ""),
        })
    return pd.DataFrame(rows)


def get_7day_forecast(forecast_url: str) -> pd.DataFrame:
    """Fetch the NWS 7-day forecast and return all periods as a DataFrame."""
    data    = _nws_get(forecast_url)
    periods = data.get("properties", {}).get("periods", [])

    rows = []
    for p in periods:
        rows.append({
            "Period":      p.get("name", ""),
            "Temp (°F)":   p.get("temperature"),
            "Wind":        p.get("windSpeed", ""),
            "Forecast":    p.get("shortForecast", ""),
            "Details":     p.get("detailedForecast", ""),
        })
    return pd.DataFrame(rows)


print("NWS API helpers defined ✓")

NWS API helpers defined ✓


In [8]:
# ── Widgets ───────────────────────────────────────────────────────────────────

W = ipw.Layout  # shorthand

address_input = ipw.Text(
    description="Address:",
    placeholder="e.g., Denver, CO  or  Houston, TX  or  90210",
    layout=W(width="420px"),
)
geocode_btn = ipw.Button(
    description="Geocode",
    button_style="primary",
    icon="search",
    layout=W(width="110px"),
)
status_bar = ipw.HTML(
    value="<b>Ready.</b> Type an address and click <i>Geocode</i>, or click anywhere on the map.",
    layout=W(width="100%"),
)
coord_label = ipw.HTML(
    value="<i style='color:#666'>No location selected yet.</i>",
    layout=W(width="100%"),
)
weather_output = ipw.Output()

# ── ipyleaflet map ────────────────────────────────────────────────────────────
# Use a permissive basemap provider to avoid OpenStreetMap tile referer blocking
tile_layer = ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.CartoDB.Positron)
m = ipyleaflet.Map(
    center=DEFAULT_CENTER,
    zoom=DEFAULT_ZOOM,
    layers=[tile_layer],
    layout=W(height="450px", width="100%"),
    scroll_wheel_zoom=True,
)

# Marker starts hidden at the default centre; shown after first selection
_marker = ipyleaflet.Marker(
    location=DEFAULT_CENTER,
    draggable=True,
    title="Selected location",
)
_marker_added = True   # track whether the marker layer is on the map

# ── Assemble the UI ───────────────────────────────────────────────────────────
top_row  = ipw.HBox([address_input, geocode_btn], layout=W(align_items="center"))
ui = ipw.VBox(
    [top_row, status_bar, coord_label, m, weather_output],
    layout=W(width="100%"),
)
display(ui)

In [8]:
# ── Weather display ───────────────────────────────────────────────────────────

def _render_current(conditions: dict, city: str, state: str) -> ipw.HTML:
    """Build an HTML card for current conditions."""
    location_str = f"{city}, {state}" if city else "Selected Location"
    temp = conditions.get("temperature_f")
    temp_s = f"{temp} °F" if temp is not None else "N/A"
    wind = conditions.get("wind_speed")
    wind_s = f"{wind:.0f} km/h" if wind is not None else "N/A"
    rh = conditions.get("humidity_pct")
    rh_s = f"{rh}%" if rh is not None else "N/A"
    desc = conditions.get("description", "N/A")
    ts = conditions.get("timestamp", "")
    if ts:
        try:
            ts = datetime.fromisoformat(ts).strftime("%Y-%m-%d %H:%M UTC")
        except ValueError:
            pass

    html = f"""
    <div style="font-family:sans-serif;border:1px solid #ccc;border-radius:8px;
                padding:16px;max-width:500px;background:#f9f9f9;">
      <h3 style="margin:0 0 8px 0">📍 {location_str}</h3>
      <p style="margin:4px 0;color:#555;font-size:0.85em">Observed: {ts}</p>
      <hr style="border:none;border-top:1px solid #ddd;margin:8px 0">
      <table style="width:100%;border-collapse:collapse;font-size:0.95em">
        <tr><td style="padding:4px 8px">🌡 Temperature</td>
            <td style="padding:4px 8px;font-weight:bold">{temp_s}</td></tr>
        <tr><td style="padding:4px 8px">💨 Wind Speed</td>
            <td style="padding:4px 8px;font-weight:bold">{wind_s}</td></tr>
        <tr><td style="padding:4px 8px">💧 Rel. Humidity</td>
            <td style="padding:4px 8px;font-weight:bold">{rh_s}</td></tr>
        <tr><td style="padding:4px 8px">☁ Conditions</td>
            <td style="padding:4px 8px;font-weight:bold">{desc}</td></tr>
      </table>
    </div>
    """
    return ipw.HTML(value=html)


def _render_hourly_chart(df: pd.DataFrame) -> ipw.Output:
    """Render a matplotlib temperature line chart inside an ipw.Output."""
    out = ipw.Output()
    with out:
        fig, ax = plt.subplots(figsize=(10, 3.5))
        ax.plot(
            df["Time"],
            df["Temp (°F)"],
            marker="o",
            markersize=4,
            linewidth=2,
            color="#1f77b4",
        )
        ax.fill_between(df["Time"], df["Temp (°F)"], alpha=0.12, color="#1f77b4")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
        plt.xticks(rotation=35, ha="right")
        ax.set_xlabel("Time (local)")
        ax.set_ylabel("Temperature (°F)")
        ax.set_title("Hourly Temperature – Next 24 Hours")
        ax.grid(True, linestyle="--", alpha=0.5)
        fig.tight_layout()
        plt.show()
    return out


def _render_weather(
    conditions: dict,
    hourly_df: pd.DataFrame,
    forecast_df: pd.DataFrame,
    city: str,
    state: str,
) -> ipw.Tab:
    """Assemble the three-tab weather panel."""
    tab = ipw.Tab()

    current_widget = _render_current(conditions, city, state)
    tab.children = [
        ipw.VBox([current_widget]),
        ipw.VBox([_render_hourly_chart(hourly_df), ipw.Output()]),
        ipw.VBox([ipw.Output()]),
    ]

    hourly_table_out = ipw.Output()
    with hourly_table_out:
        display(hourly_df.style.hide(axis="index"))
    tab.children[1].children = (*tab.children[1].children[:-1], hourly_table_out)

    forecast_table_out = ipw.Output()
    with forecast_table_out:
        display(forecast_df.style.hide(axis="index"))
    tab.children[2].children = (forecast_table_out,)

    tab.set_title(0, "Current Conditions")
    tab.set_title(1, "Next 24 Hours")
    tab.set_title(2, "7-Day Forecast")
    return tab


def _format_coordinate(value: float, positive_label: str, negative_label: str) -> str:
    suffix = positive_label if value >= 0 else negative_label
    return f"{abs(value):.5f}°{suffix}"


# ── Core update logic ─────────────────────────────────────────────────────────

def fetch_and_display_weather(lat: float, lon: float) -> bool:
    """Pull all NWS data for (lat, lon) and render it in weather_output."""
    weather_output.clear_output(wait=True)
    with weather_output:
        display(ipw.HTML("<i>Fetching weather data...</i>"))

    try:
        pts = get_nws_points(lat, lon)
        cond = get_current_conditions(pts["observationStations"])
        hourly = get_hourly_forecast(pts["forecastHourly"])
        forecast = get_7day_forecast(pts["forecast"])
        tab = _render_weather(cond, hourly, forecast, pts["city"], pts["state"])
        weather_output.clear_output(wait=True)
        with weather_output:
            display(tab)
        return True
    except ValueError as exc:
        status_bar.value = f"<b style='color:orange'>Warning:</b> {exc}"
        weather_output.clear_output()
        return False
    except requests.HTTPError as exc:
        status_code = exc.response.status_code if exc.response is not None else "unknown"
        reason = exc.response.reason if exc.response is not None else str(exc)
        status_bar.value = (
            f"<b style='color:red'>NWS API error:</b> {status_code} - {reason}"
        )
        weather_output.clear_output()
        return False
    except requests.RequestException as exc:
        status_bar.value = f"<b style='color:red'>Network error:</b> {exc}"
        weather_output.clear_output()
        return False


def update_location(lat: float, lon: float) -> None:
    """Move the marker, update coord label, and kick off a weather fetch."""
    global _marker_added

    _marker.location = (lat, lon)
    if not _marker_added:
        m.add(_marker)
        _marker_added = True

    m.center = (lat, lon)
    m.zoom = 10
    coord_label.value = (
        f"<b>Selected:</b> "
        f"{_format_coordinate(lat, 'N', 'S')}, {_format_coordinate(lon, 'E', 'W')}"
    )
    status_bar.value = (
        f"<b>Location set.</b> Fetching weather for ({lat:.4f}, {lon:.4f})..."
    )

    success = fetch_and_display_weather(lat, lon)
    if success:
        status_bar.value = "<b>Done.</b> Click the map or geocode a new address to refresh."


# ── Button callback ───────────────────────────────────────────────────────────

def on_geocode_clicked(_=None) -> None:
    address = address_input.value.strip()
    if not address:
        status_bar.value = "<b style='color:orange'>Warning:</b> Please enter an address."
        return

    geocode_btn.disabled = True
    status_bar.value = f"<i>Geocoding <b>{address}</b>...</i>"
    try:
        lat, lon = geocode_address(address)
        update_location(lat, lon)
    except ValueError as exc:
        status_bar.value = f"<b style='color:orange'>Warning:</b> {exc}"
    except Exception as exc:
        status_bar.value = f"<b style='color:red'>Unexpected error:</b> {exc}"
    finally:
        geocode_btn.disabled = False


geocode_btn.on_click(on_geocode_clicked)
address_input.on_submit(on_geocode_clicked)


# ── Map click callback ────────────────────────────────────────────────────────

def handle_map_interaction(**kwargs) -> None:
    if kwargs.get("type") != "click":
        return

    coords = kwargs.get("coordinates")
    if not coords or len(coords) < 2:
        return

    lat, lon = coords[0], coords[1]
    address_input.value = ""
    update_location(lat, lon)


m.on_interaction(handle_map_interaction)

print("Callbacks wired. The UI above is ready to use.")

Callbacks wired. The UI above is ready to use.


In [9]:
# ── Smoke test ────────────────────────────────────────────────────────────────
# Validates geocoding + NWS /points with a known US location (Denver, CO).
# Run this cell independently to confirm the full pipeline before using the UI.

_test_address = "Denver, CO"
_lat, _lon = geocode_address(_test_address)
assert abs(_lat - 39.7) < 0.5,  f"Unexpected latitude: {_lat}"
assert abs(_lon - (-104.9)) < 0.5, f"Unexpected longitude: {_lon}"
print(f"Geocode ✓  {_test_address} → ({_lat:.4f}, {_lon:.4f})")

_pts = get_nws_points(_lat, _lon)
assert _pts.get("forecast"),       "Missing 'forecast' URL from NWS /points"
assert _pts.get("forecastHourly"), "Missing 'forecastHourly' URL from NWS /points"
assert _pts.get("observationStations"), "Missing 'observationStations' URL from NWS /points"
print(f"NWS /points ✓  city={_pts['city']}, state={_pts['state']}")

_hourly = get_hourly_forecast(_pts["forecastHourly"])
assert len(_hourly) == 24, f"Expected 24 hourly rows, got {len(_hourly)}"
print(f"Hourly forecast ✓  {len(_hourly)} periods")

_daily = get_7day_forecast(_pts["forecast"])
assert len(_daily) >= 7, f"Expected ≥7 forecast rows, got {len(_daily)}"
print(f"7-day forecast ✓  {len(_daily)} periods")

print("\nAll validation checks passed ✓")

Geocode ✓  Denver, CO → (39.7392, -104.9849)
NWS /points ✓  city=Glendale, state=CO
Hourly forecast ✓  24 periods
7-day forecast ✓  14 periods

All validation checks passed ✓


In [1]:
import pytest

In [2]:
# Run the local pytest suite for the WeatherData helper
import subprocess
import sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_weather_data.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)


....                                                                     [100%]
4 passed in 1.93s

